In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType,StructField, StringType, IntegerType

spark = SparkSession.builder.appName("Jupyter").getOrCreate()

conf = spark.sparkContext.getConf()

# Filter and print configurations that start with 'spark.sql'
print("Spark SQL Configurations:")
for key, value in conf.getAll():
    if key.startswith("spark.sql"):
        print(f"{key} = {value}")

print(f"spark.sql.catalog.data.s3.endpoint: {spark.conf.get('spark.sql.catalog.data.s3.endpoint', 'Not Set')}")

for key, value in conf.getAll():
    if key.startswith("spark.hadoop"):
        print(f"{key} = {value}")

for key, value in conf.getAll():
    print(f"{key} = {value}")
spark

Spark SQL Configurations:
spark.sql.catalog.iceberg.warehouse = s3://kxu-iceberg-data/cdc-bronze-data
spark.sql.catalog.data.s3.endpoint = http://minio-s3:9000
spark.sql.defaultCatalog = data
spark.sql.catalog.kxudata.type = hadoop
spark.sql.catalog.data.jdbc.password = kxuiceberg
spark.sql.catalog.data.io-impl = org.apache.iceberg.aws.s3.S3FileIO
spark.sql.catalog.data = org.apache.iceberg.spark.SparkCatalog
spark.sql.catalog.iceberg.catalog-impl = org.apache.iceberg.jdbc.JdbcCatalog
spark.sql.catalog.data.catalog-impl = org.apache.iceberg.jdbc.JdbcCatalog
spark.sql.catalog.iceberg.uri = jdbc:postgresql://pg-catalog:5432/kxuiceberg
spark.sql.catalog.iceberg.jdbc.password = kxuiceberg
spark.sql.catalog.data.warehouse = s3://kxu-iceberg-data/bronze_data
spark.sql.catalog.data.jdbc.user = kxuiceberg
spark.sql.catalogImplementation = in-memory
spark.sql.catalog.iceberg.io-impl = org.apache.iceberg.aws.s3.S3FileIO
spark.sql.catalog.kxudata.warehouse = /home/iceberg/warehouse
spark.sql.cata

In [2]:
namespace1 = "db"
table1 = f"{namespace1}.test"
namespace2 = "cataglog"
table2 = f"{namespace1}.catalog_test"


In [ ]:
# only required for nessie catalog, which requires namespace to be created explicitly
spark.sql("CREATE NAMESPACE if Not exists db")
spark.sql("CREATE NAMESPACE if Not exists catalog")
spark.sql("CREATE NAMESPACE if Not exists catalog_test")

In [ ]:
spark.sql("CREATE NAMESPACE if Not exists silver_data.db")

In [3]:
data = [("James","","Smith","36636","M",3000),
    ("Michael","Rose","","40288","M",4000),
    ("Robert","","Williams","42114","M",4000),
    ("Maria","Anne","Jones","39192","F",4000),
    ("Jen","Mary","Brown","","F",-1)
  ]

schema = StructType([ \
    StructField("firstname",StringType(),True), \
    StructField("middlename",StringType(),True), \
    StructField("lastname",StringType(),True), \
    StructField("id", StringType(), True), \
    StructField("gender", StringType(), True), \
    StructField("salary", IntegerType(), True) \
  ])

data

df = spark.createDataFrame(data=data, schema=schema)
df.printSchema()


root
 |-- firstname: string (nullable = true)
 |-- middlename: string (nullable = true)
 |-- lastname: string (nullable = true)
 |-- id: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- salary: integer (nullable = true)



In [4]:
df.writeTo(table1).createOrReplace()

In [5]:
df.writeTo(table2).createOrReplace()

In [6]:
spark.sql("CREATE TABLE data.db.my_new_table ( \
  id INT, \
  name STRING\
) USING iceberg")

DataFrame[]

In [ ]:
spark.sql("CREATE TABLE iceberg.rpc.my_new_table ( \
  id INT, \
  name STRING\
) USING iceberg")

In [14]:
import textwrap

ddl = spark.sql("show create table iceberg.rpc.pizzas").collect()[0][0]
print("\n".join(textwrap.wrap(ddl, width=100)))

CREATE TABLE iceberg.rpc.pizzas (   store_id BIGINT,   meats STRING,   sauce STRING,   veggies
STRING,   date_ordered STRING,   order_id STRING,   cheese STRING) USING iceberg LOCATION
's3a://kxu-iceberg-data/cdc-bronze-data/rpc/pizzas' TBLPROPERTIES (   'current-snapshot-id' =
'4533687527899206333',   'format' = 'iceberg/parquet',   'format-version' = '2',
'write.parquet.compression-codec' = 'zstd')


In [18]:
spark.sql("CREATE TABLE iceberg.rpc.pizzas_by_hour (   store_id BIGINT,   meats STRING,   sauce STRING,   veggies STRING,   date_ordered TIMESTAMP,   order_id STRING,   cheese STRING) PARTITIONED BY (store_id, hours(date_ordered))")

DataFrame[]

In [4]:
spark.sql("select * from iceberg.rpc.pizzas limit 10").show()


+--------+--------------------+-------+--------------------+-------------------+--------------------+------------+
|store_id|               meats|  sauce|             veggies|       date_ordered|            order_id|      cheese|
+--------+--------------------+-------+--------------------+-------------------+--------------------+------------+
|      65|     bacon & sausage|  light|     onions & tomato|2025-03-16 16:22:11|14114741144202479...|       extra|
|      65|             sausage|alfredo|  olives & pineapple|2025-03-16 16:22:11|14114741144202479...|        none|
|      65|                none|  extra|onions & mushroom...|2025-03-16 16:22:11|14114741144202479...| goat cheese|
|      65|  salami & pepperoni|  extra|              tomato|2025-03-16 16:22:11|14114741144202479...|three cheese|
|      30|                 ham|  extra|              tomato|2025-03-16 16:22:21|15485239504293605...| goat cheese|
|      30|                 ham|    bbq|mushrooms & pinea...|2025-03-16 16:22:21|

In [5]:
table1 = "iceberg.rpc.pizzas"

In [ ]:
res = spark.sql(f"SELECT * FROM {table1}")
print(f"schema: {res.schema}")
res.show()

In [ ]:
res = spark.sql(f"SELECT * FROM {table2}")
res.show()

In [14]:
df.writeTo("silver_data.db.test2").createOrReplace()

In [15]:
spark.sql("select * from silver_data.db.test2").show()

+---------+----------+--------+-----+------+------+
|firstname|middlename|lastname|   id|gender|salary|
+---------+----------+--------+-----+------+------+
|    James|          |   Smith|36636|     M|  3000|
|  Michael|      Rose|        |40288|     M|  4000|
|   Robert|          |Williams|42114|     M|  4000|
|    Maria|      Anne|   Jones|39192|     F|  4000|
|      Jen|      Mary|   Brown|     |     F|    -1|
+---------+----------+--------+-----+------+------+



In [ ]:
spark.sql("CREATE BRANCH ke_test")

In [6]:
#Inspecting the history of the table:
res = spark.sql(f"SELECT made_current_at, snapshot_id, parent_id, is_current_ancestor FROM {table1}.history")
res.show(truncate=False)

+-----------------------+-------------------+-------------------+-------------------+
|made_current_at        |snapshot_id        |parent_id          |is_current_ancestor|
+-----------------------+-------------------+-------------------+-------------------+
|2025-03-16 23:12:01.434|5686744723707271212|NULL               |true               |
|2025-03-16 23:17:01.604|2010242271663896358|5686744723707271212|true               |
|2025-03-16 23:22:01.701|5799891980476499415|2010242271663896358|true               |
|2025-03-16 23:27:02.101|1684896047087981051|5799891980476499415|true               |
+-----------------------+-------------------+-------------------+-------------------+



In [7]:
#inspect the table snapshots
res = spark.sql(f"SELECT committed_at, snapshot_id, operation, manifest_list, summary FROM {table1}.snapshots")
res.show(truncate=False)


+-----------------------+-------------------+---------+-------------------------------------------------------------------------------------------------------------------------------+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|committed_at           |snapshot_id        |operation|manifest_list                                                                                                                  |summary                                                                                                                                                                                               

In [8]:
#query to get back the data files
res = spark.sql(f"SELECT file_path, file_format, record_count FROM {table1}.files")
res.show(truncate=False)

+-----------------------------------------------------------------------------------------------------------------------------+-----------+------------+
|file_path                                                                                                                    |file_format|record_count|
+-----------------------------------------------------------------------------------------------------------------------------+-----------+------------+
|s3a://kxu-iceberg-data/cdc-bronze-data/rpc/pizzas/data/00001-1742167631747-ba0696df-f720-491d-b7d9-6138e3d7339b-00001.parquet|PARQUET    |91          |
|s3a://kxu-iceberg-data/cdc-bronze-data/rpc/pizzas/data/00001-1742167331417-5c19022e-0890-46c9-8574-a7aba7fbb117-00001.parquet|PARQUET    |103         |
|s3a://kxu-iceberg-data/cdc-bronze-data/rpc/pizzas/data/00001-1742167031076-c924919e-5eef-48c4-8cff-100785462de6-00001.parquet|PARQUET    |89          |
|s3a://kxu-iceberg-data/cdc-bronze-data/rpc/pizzas/data/00001-1742166730751-8c349f

In [9]:
#query the manifests
res = spark.sql(f"SELECT length, path, added_data_files_count, added_snapshot_id FROM {table1}.manifests")
res.show(truncate=False)

+------+-------------------------------------------------------------------------------------------------------+----------------------+-------------------+
|length|path                                                                                                   |added_data_files_count|added_snapshot_id  |
+------+-------------------------------------------------------------------------------------------------------+----------------------+-------------------+
|7155  |s3a://kxu-iceberg-data/cdc-bronze-data/rpc/pizzas/metadata/63d2925c-9e83-4a47-b3ab-151062e8d9ee-m0.avro|1                     |3109283121833817178|
|7152  |s3a://kxu-iceberg-data/cdc-bronze-data/rpc/pizzas/metadata/932f23b1-0627-4a9a-9492-50b899fa5d86-m0.avro|1                     |1684896047087981051|
|7159  |s3a://kxu-iceberg-data/cdc-bronze-data/rpc/pizzas/metadata/1cd2f2e9-4c52-426f-b995-b6a6c7959091-m0.avro|1                     |5799891980476499415|
|7154  |s3a://kxu-iceberg-data/cdc-bronze-data/rpc/pizzas/metada

In [10]:
#query the metadata log entries
res = spark.sql(f"SELECT timestamp, file, latest_snapshot_id, latest_schema_id, latest_sequence_number FROM {table1}.metadata_log_entries")
res.show(truncate=False)

+-----------------------+-------------------------------------------------------------------------------------------------------------------+-------------------+----------------+----------------------+
|timestamp              |file                                                                                                               |latest_snapshot_id |latest_schema_id|latest_sequence_number|
+-----------------------+-------------------------------------------------------------------------------------------------------------------+-------------------+----------------+----------------------+
|2025-03-16 23:07:20.552|s3a://kxu-iceberg-data/cdc-bronze-data/rpc/pizzas/metadata/00000-fd34dbc1-4b80-4497-89a7-3c9fc57fd5a3.metadata.json|NULL               |NULL            |NULL                  |
|2025-03-16 23:12:01.434|s3a://kxu-iceberg-data/cdc-bronze-data/rpc/pizzas/metadata/00001-dec83107-5305-4b7b-96b8-026a47038dc4.metadata.json|5686744723707271212|0               |1             

In [ ]:
spark.stop()